# Imports

In [1]:
import pandas as pd
import numpy as np
import os
import glob

data_path = '/kaggle/input/datasets/hayeonchung353/song-lyrics-data/'

all_files = glob.glob(data_path + '*.csv')

all_files = [f for f in all_files if 'BTS.csv' not in f and 'CardiB.csv' not in f and 'Drake.csv' not in f and 'NickiMinaj.csv' not in f and 'Eminem.csv' not in f]

print(f"Found {len(all_files)} CSV files (excluding BTS, Cardi B, Drake, Nicki Minaj, and Eminem)")

dfs = []

for f in all_files:
    try:
        df = pd.read_csv(f, encoding='utf-8', on_bad_lines='skip')
        df['source_file'] = os.path.basename(f)
        dfs.append(df)
    except Exception as e:
        print(f"Skipped {f}: {e}")

raw = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(dfs)} files successfully")
print("Columns found:", raw.columns.tolist())

raw.head()

Found 16 CSV files (excluding BTS, Cardi B, Drake, Nicki Minaj, and Eminem)
Loaded 16 files successfully
Columns found: ['Unnamed: 0', 'Artist', 'Title', 'Album', 'Year', 'Date', 'Lyric', 'source_file']


,Unnamed: 0,Artist,Title,Album,Year,Date,Lyric,source_file
0,0.0,Coldplay,The Scientist,A Rush of Blood to the Head,2002.0,2002-08-26,come up to meet you tell you i'm sorry you don...,ColdPlay.csv
1,1.0,Coldplay,Viva la Vida,Viva La Vida or Death and All His Friends,2008.0,2008-05-25,chris martin i used to rule the world seas wou...,ColdPlay.csv
2,2.0,Coldplay,Fix You,X&Y,2005.0,2005-06-06,chris martin when you try your best but you do...,ColdPlay.csv
3,3.0,Coldplay,Yellow,Parachutes,2000.0,2000-06-26,chris martin look at the stars look how they s...,ColdPlay.csv
4,4.0,Coldplay,Hymn for the Weekend,A Head Full of Dreams,2016.0,2016-01-25,beyoncé and said drink from me drink from me o...,ColdPlay.csv


# Detect Key Columns + Clean Lyrics

In [2]:
lyrics_col = None
for candidate in ['Lyric', 'lyric', 'Lyrics', 'lyrics', 'text', 'Text']:
    if candidate in raw.columns:
        lyrics_col = candidate
        break

title_col = None
for candidate in ['Title', 'title', 'Song', 'song', 'Track', 'track']:
    if candidate in raw.columns:
        title_col = candidate
        break

album_col = None
for candidate in ['Album', 'album']:
    if candidate in raw.columns:
        album_col = candidate
        break

year_col = None
for candidate in ['Year', 'year', 'Date', 'date']:
    if candidate in raw.columns:
        year_col = candidate
        break

artist_col = None
for candidate in ['Artist', 'artist']:
    if candidate in raw.columns:
        artist_col = candidate
        break

if artist_col is None:
    artist_col = 'artist_file'

print("Lyrics column:", lyrics_col)
print("Title column:", title_col)
print("Album column:", album_col)
print("Year column:", year_col)
print("Artist column:", artist_col)

raw['lyrics_clean'] = raw[lyrics_col].astype(str).str.strip()
raw = raw[raw['lyrics_clean'].notna()]
raw = raw[raw['lyrics_clean'].str.len() > 0].copy()
raw['Year'] = raw['Year'].astype('Int64')

raw['doc_id'] = range(len(raw))
raw['doc_length_chars'] = raw['lyrics_clean'].str.len()
raw['doc_length_words'] = raw['lyrics_clean'].str.split().str.len()

raw = raw[~raw['Lyric'].astype(str).str.contains(r'[\uac00-\ud7a3]', regex=True, na=False)].copy()
print("Rows after dropping Korean lyrics:", len(raw))

raw.head()

Lyrics column: Lyric
Title column: Title
Album column: Album
Year column: Year
Artist column: Artist
Rows after dropping Korean lyrics: 4363


,Unnamed: 0,Artist,Title,Album,Year,Date,Lyric,source_file,lyrics_clean,doc_id,doc_length_chars,doc_length_words
0,0.0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,2002-08-26,come up to meet you tell you i'm sorry you don...,ColdPlay.csv,come up to meet you tell you i'm sorry you don...,0,927,189
1,1.0,Coldplay,Viva la Vida,Viva La Vida or Death and All His Friends,2008,2008-05-25,chris martin i used to rule the world seas wou...,ColdPlay.csv,chris martin i used to rule the world seas wou...,1,1799,346
2,2.0,Coldplay,Fix You,X&Y,2005,2005-06-06,chris martin when you try your best but you do...,ColdPlay.csv,chris martin when you try your best but you do...,2,973,191
3,3.0,Coldplay,Yellow,Parachutes,2000,2000-06-26,chris martin look at the stars look how they s...,ColdPlay.csv,chris martin look at the stars look how they s...,3,1050,222
4,4.0,Coldplay,Hymn for the Weekend,A Head Full of Dreams,2016,2016-01-25,beyoncé and said drink from me drink from me o...,ColdPlay.csv,beyoncé and said drink from me drink from me o...,4,1685,359


In [3]:
# Drop rows where lyrics are not in English
def is_mostly_english(text, threshold=0.9):
    text = str(text)
    if len(text) == 0:
        return False
    
    english_chars = sum(c.isascii() for c in text)
    return (english_chars / len(text)) >= threshold

raw = raw[raw['Lyric'].apply(is_mostly_english)].copy()

print("Rows after keeping mostly English lyrics:", len(raw))

Rows after keeping mostly English lyrics: 4356


In [4]:
# Drop placeholder rows where lyrics were not yet available
raw = raw[~raw[lyrics_col].str.contains('lyrics for this song have yet to be released', 
                                          case=False, na=False)].copy()

print(f"After dropping unreleased placeholders: {len(raw)} rows")

After dropping unreleased placeholders: 4141 rows


# LIB Table

In [5]:
LIB = raw.copy()

keep_cols = ['doc_id', 'source_file', artist_col, title_col, album_col, year_col,
             'doc_length_chars', 'doc_length_words', 'lyrics_clean']

keep_cols = [col for col in keep_cols if col is not None and col in LIB.columns]

LIB = LIB[keep_cols].copy()

LIB = LIB.rename(columns={
    artist_col: 'artist',
    title_col: 'title',
    album_col: 'album',
    year_col: 'year',
    'lyrics_clean': 'lyrics'
})

LIB.to_csv('LIB.csv', index=False)

print("LIB observations:", len(LIB))
print("LIB columns:", LIB.columns.tolist())
print("Average document length in characters:", round(LIB['doc_length_chars'].mean(), 2))

LIB.head()

LIB observations: 4141
LIB columns: ['doc_id', 'source_file', 'artist', 'title', 'album', 'year', 'doc_length_chars', 'doc_length_words', 'lyrics']
Average document length in characters: 1632.16


,doc_id,source_file,artist,title,album,year,doc_length_chars,doc_length_words,lyrics
0,0,ColdPlay.csv,Coldplay,The Scientist,A Rush of Blood to the Head,2002,927,189,come up to meet you tell you i'm sorry you don...
1,1,ColdPlay.csv,Coldplay,Viva la Vida,Viva La Vida or Death and All His Friends,2008,1799,346,chris martin i used to rule the world seas wou...
2,2,ColdPlay.csv,Coldplay,Fix You,X&Y,2005,973,191,chris martin when you try your best but you do...
3,3,ColdPlay.csv,Coldplay,Yellow,Parachutes,2000,1050,222,chris martin look at the stars look how they s...
4,4,ColdPlay.csv,Coldplay,Hymn for the Weekend,A Head Full of Dreams,2016,1685,359,beyoncé and said drink from me drink from me o...


# CORPUS Table

In [6]:
import re

def tokenize_lyrics(text):
    lines = str(text).splitlines()
    rows = []

    for line_num, line in enumerate(lines):
        # Use [a-z'] instead of \w to match only ASCII letters and apostrophes from appearing as DTM columns.
        tokens = re.findall(r"\b[a-z][a-z']*[a-z]\b|\b[a-z]\b", line.lower())

        for token_num, token in enumerate(tokens):
            rows.append({
                'line_num': line_num,
                'token_num': token_num,
                'token_str': token,
                'term_str': token
            })

    return rows

corpus_rows = []

for _, row in LIB.iterrows():
    token_rows = tokenize_lyrics(row['lyrics'])

    for token_row in token_rows:
        token_row['doc_id'] = row['doc_id']
        token_row['artist'] = row.get('artist', None)
        token_row['title'] = row.get('title', None)
        token_row['album'] = row.get('album', None)
        token_row['year'] = row.get('year', None)
        corpus_rows.append(token_row)

CORPUS = pd.DataFrame(corpus_rows)

CORPUS = CORPUS[['doc_id', 'artist', 'title', 'album', 'year',
                 'line_num', 'token_num', 'token_str', 'term_str']]

CORPUS.to_csv('CORPUS.csv', index=False)

print("CORPUS observations:", len(CORPUS))
print("CORPUS columns:", CORPUS.columns.tolist())
print("OHCO structure: doc_id, line_num, token_num")

CORPUS.head()

CORPUS observations: 1376718
CORPUS columns: ['doc_id', 'artist', 'title', 'album', 'year', 'line_num', 'token_num', 'token_str', 'term_str']
OHCO structure: doc_id, line_num, token_num


,doc_id,artist,title,album,year,line_num,token_num,token_str,term_str
0,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,0,come,come
1,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,1,up,up
2,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,2,to,to
3,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,3,meet,meet
4,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,4,you,you


# Adding POS and POS Group to CORPUS

In [7]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')


tokens = CORPUS['term_str'].tolist()

try:
    tagged = nltk.pos_tag(tokens)
except LookupError:
    tagged = nltk.pos_tag(tokens, lang='eng')

CORPUS['pos'] = [tag for word, tag in tagged]

def get_pos_group(pos):
    if pos.startswith('N'):
        return 'noun'
    elif pos.startswith('V'):
        return 'verb'
    elif pos.startswith('J'):
        return 'adj'
    elif pos.startswith('R'):
        return 'adv'
    else:
        return 'other'

CORPUS['pos_group'] = CORPUS['pos'].apply(get_pos_group)
CORPUS.to_csv('CORPUS.csv', index=False)

CORPUS.head()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


,doc_id,artist,title,album,year,line_num,token_num,token_str,term_str,pos,pos_group
0,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,0,come,come,VB,verb
1,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,1,up,up,RP,adv
2,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,2,to,to,TO,other
3,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,3,meet,meet,VB,verb
4,0,Coldplay,The Scientist,A Rush of Blood to the Head,2002,0,4,you,you,PRP,other


# VOCAB Table

In [8]:
VOCAB = CORPUS.groupby('term_str').agg(
    n=('term_str', 'count'),
    p=('term_str', lambda x: len(x) / len(CORPUS)),
    i=('doc_id', 'nunique')
).reset_index()

VOCAB['dfidf'] = VOCAB['i'] * np.log2(len(LIB) / VOCAB['i'])

max_pos = (
    CORPUS.groupby('term_str')['pos']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={'pos': 'max_pos'})
)

max_pos_group = (
    CORPUS.groupby('term_str')['pos_group']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={'pos_group': 'max_pos_group'})
)

VOCAB = VOCAB.merge(max_pos, on='term_str', how='left')
VOCAB = VOCAB.merge(max_pos_group, on='term_str', how='left')

# Add porter_stem column 
from nltk.stem import PorterStemmer
ps = PorterStemmer()
VOCAB['porter_stem'] = VOCAB['term_str'].apply(ps.stem)

# Drop any non-ASCII terms that slipped through
VOCAB = VOCAB[VOCAB['term_str'].str.match(r"^[a-z'][a-z']*$")].copy()

stop_words = set(
    "a an and are as at be by for from has he in is it its of on that the to "
    "was were will with i you me my we our your they them their this those "
    "these there here but or if then so".split()
)

VOCAB['stop'] = VOCAB['term_str'].isin(stop_words)

VOCAB = VOCAB.sort_values('dfidf', ascending=False)

VOCAB.to_csv('VOCAB.csv', index=False)

print("VOCAB observations:", len(VOCAB))
print("VOCAB columns:", VOCAB.columns.tolist())

VOCAB.head(20)

VOCAB observations: 20583
VOCAB columns: ['term_str', 'n', 'p', 'i', 'dfidf', 'max_pos', 'max_pos_group', 'porter_stem', 'stop']


,term_str,n,p,i,dfidf,max_pos,max_pos_group,porter_stem,stop
18304,time,3910,0.002840,1519,2197.776289,NN,noun,time,False
15775,see,4204,0.003054,1538,2197.684650,VBP,verb,see,False
12253,not,4454,0.003235,1507,2197.657777,RB,adv,not,False
12563,one,5271,0.003829,1459,2195.794007,CD,other,one,False
20455,you're,5987,0.004349,1590,2195.714375,NN,noun,you'r,False
1117,baby,8018,0.005824,1593,2195.525067,NN,noun,babi,False
15585,say,4002,0.002907,1447,2194.974964,VBP,verb,say,False
12045,never,4473,0.003249,1440,2194.430963,RB,adv,never,False
7257,get,5405,0.003926,1609,2194.378137,VB,verb,get,False
2616,can,5515,0.004006,1634,2192.127372,MD,other,can,False


In [9]:
top_20_significant_words = (
    VOCAB[VOCAB['stop'] == False]
    .sort_values('dfidf', ascending=False)
    .head(20)
)

top_20_significant_words[['term_str', 'n', 'i', 'dfidf', 'max_pos', 'max_pos_group']]

,term_str,n,i,dfidf,max_pos,max_pos_group
18304,time,3910,1519,2197.776289,NN,noun
15775,see,4204,1538,2197.684650,VBP,verb
12253,not,4454,1507,2197.657777,RB,adv
12563,one,5271,1459,2195.794007,CD,other
20455,you're,5987,1590,2195.714375,NN,noun
1117,baby,8018,1593,2195.525067,NN,noun
15585,say,4002,1447,2194.974964,VBP,verb
12045,never,4473,1440,2194.430963,RB,adv
7257,get,5405,1609,2194.378137,VB,verb
2616,can,5515,1634,2192.127372,MD,other


# BOW Table

In [10]:
BOW = (
    CORPUS.groupby(['doc_id', 'term_str'])
    .size()
    .reset_index(name='n')
)

doc_lengths = CORPUS.groupby('doc_id').size().reset_index(name='doc_n')

BOW = BOW.merge(doc_lengths, on='doc_id', how='left')
BOW['tf'] = BOW['n'] / BOW['doc_n']

# Keep only terms that survived VOCAB ASCII filtering (prevents non-English DTM columns)
clean_terms = set(VOCAB['term_str'])
BOW = BOW[BOW['term_str'].isin(clean_terms)].copy()

BOW = BOW.merge(VOCAB[['term_str', 'i']], on='term_str', how='left')
BOW['idf'] = np.log2(len(LIB) / BOW['i'])
BOW['tfidf'] = BOW['tf'] * BOW['idf']

BOW.to_csv('BOW.csv', index=False)

print("BOW observations:", len(BOW))
print("BOW columns:", BOW.columns.tolist())
print("Bag level: doc_id")

BOW.head()

BOW observations: 435865
BOW columns: ['doc_id', 'term_str', 'n', 'doc_n', 'tf', 'i', 'idf', 'tfidf']
Bag level: doc_id


,doc_id,term_str,n,doc_n,tf,i,idf,tfidf
0,0,a,3,189,0.015873,3361,0.301089,0.004779
1,0,ahooh,1,189,0.005291,5,9.693835,0.051290
2,0,and,5,189,0.026455,3569,0.214459,0.005674
3,0,apart,3,189,0.015873,155,4.739639,0.075232
4,0,are,2,189,0.010582,1241,1.738476,0.018397


# DTM Table

In [11]:
DTM = BOW.pivot_table(
    index='doc_id',
    columns='term_str',
    values='n',
    fill_value=0
)

DTM.to_csv('DTM.csv')

print("DTM shape:", DTM.shape)
DTM.head()

DTM shape: (4141, 20583)


term_str,a,a'bang,a'gwaan,aa,aaaa,aaaaaaah,aaaahh,aaaayo,aaagain,aaah,...,zro,zu,zub,zuboomboom,zucchinis,zurich,zuse,zuzu,zwrotka,zz
doc_id,,,,,,,,,,,,,,,,,,,,,
0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# TFIDF Table

In [12]:
TFIDF = BOW.pivot_table(
    index='doc_id',
    columns='term_str',
    values='tfidf',
    fill_value=0
)

TFIDF.to_csv('TFIDF.csv')

print("TFIDF shape:", TFIDF.shape)
TFIDF.head()

TFIDF shape: (4141, 20583)


term_str,a,a'bang,a'gwaan,aa,aaaa,aaaaaaah,aaaahh,aaaayo,aaagain,aaah,...,zro,zu,zub,zuboomboom,zucchinis,zurich,zuse,zuzu,zwrotka,zz
doc_id,,,,,,,,,,,,,,,,,,,,,
0,0.004779,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.005221,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.008138,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.007676,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
print("TFIDF formula: tfidf = term frequency * inverse document frequency")
print("tf = count of term in document / total number of terms in document")
print("idf = log2(number of documents / number of documents containing the term)")

TFIDF formula: tfidf = term frequency * inverse document frequency
tf = count of term in document / total number of terms in document
idf = log2(number of documents / number of documents containing the term)


# Reduced + Normalized TFIDF_L2 Table

In [14]:
from sklearn.preprocessing import normalize

num_features = 1000

sig_terms = (
    VOCAB[VOCAB['stop'] == False]
    .sort_values('dfidf', ascending=False)
    .head(num_features)['term_str']
    .tolist()
)

TFIDF_REDUCED = TFIDF[sig_terms].copy()

TFIDF_L2_array = normalize(TFIDF_REDUCED, norm='l2', axis=1)

TFIDF_L2 = pd.DataFrame(
    TFIDF_L2_array,
    index=TFIDF_REDUCED.index,
    columns=TFIDF_REDUCED.columns
)

TFIDF_L2.to_csv('TFIDF_L2.csv')

print("TFIDF_L2 shape:", TFIDF_L2.shape)
print("Number of significant words:", len(sig_terms))
print("Principle of significant word selection: top non-stopword terms by DFIDF")

TFIDF_L2.head()

TFIDF_L2 shape: (4141, 1000)
Number of significant words: 1000
Principle of significant word selection: top non-stopword terms by DFIDF


term_str,time,see,not,one,you're,baby,say,never,get,can,...,bottle,shy,spell,here's,secret,stare,moved,pressure,list,c'mon
doc_id,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.030238,0.062413,0.000000,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.015274,0.000000,0.0,0.0,0.061862,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.027254,0.000000,0.051618,0.0,0.0,0.056961,0.025489,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.008623,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
